# 16 · Local-first web application

**Goal:** inspect the v0.14 browser entry point and verify that its operations use the same versioned HTTP API and underlying catalog services. All data below are fictional test records. No network listener is started.

In [ ]:
from fastapi.testclient import TestClient
from savoroute.api.app import create_app
from savoroute.data.catalog import load_catalog
from savoroute.services.discovery import DiscoveryService
from savoroute.retrieval.lexical import SearchFilters

In [ ]:
app = create_app()
with TestClient(app) as client:
    home = client.get("/")
    meta = client.get("/v1/meta").json()
    assert home.status_code == 200 and "text/html" in home.headers["content-type"]
    assert "script-src 'self'" in home.headers["content-security-policy"]
    assert client.get("/assets/app.js").status_code == 200
print("Web interface packaged:", home.status_code)
print("Dataset:", meta["dataset_name"], "| records:", meta["catalog_items"])

## Catalog browsing via HTTP v1

The browser calls the existing `/v1/items` endpoint. Every returned restaurant or recipe must correspond to an actual catalog record.

In [ ]:
with TestClient(app) as client:
    catalog_page = client.get("/v1/items", params={"kind":"restaurant","limit":4}).json()
assert catalog_page["total"] == 6
for item in catalog_page["items"]:
    print(item["title"], "|", item["source"])

## Search and grounded recommendations

Both routes use `DiscoveryService`. Comparing their returned catalog IDs illustrates service parity, not real-world retrieval accuracy.

In [ ]:
payload = {"query":"spicy tofu noodles","filters":{"kind":"recipe"},"limit":3}
with TestClient(app) as client:
    search = client.post("/v1/search",json=payload).json()
    recommendations = client.post("/v1/recommend",json={**payload,"preferences":{"cuisines":["Japanese"],"preference_strength":0.5}}).json()
service_ids = [h.item.item_id for h in DiscoveryService(load_catalog()).search("spicy tofu noodles",limit=3,filters=SearchFilters(kind="recipe"))]
assert [hit["item"]["item_id"] for hit in search["results"]] == service_ids
print("Search:",service_ids)
print("Recommendations:",[(r["title"],r["source"]) for r in recommendations["results"]])

## Run the actual browser UI

From a terminal, run `savoroute web serve --host 127.0.0.1 --port 8000`, then visit `http://127.0.0.1:8000/`. To browse the frozen real OSM catalog, restart the server with `--catalog data/generated/real-pilot/catalog.json`. See `docs/web-app.md`. **Do not confuse the fictional notebook examples with real-world evaluation.**